In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)


c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain_core.messages import HumanMessage
response = model.invoke([HumanMessage(content="What is the capital of France?")])
print(response.content)

The capital of France is Paris.


In [6]:
from langchain_core.messages import HumanMessage,AIMessage
model.invoke([HumanMessage("Hi"),
  AIMessage("Hello!"),
  HumanMessage("What did you just say?")])

AIMessage(content='I said "Hello!"', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 54, 'total_tokens': 60, 'completion_time': 0.017414088, 'completion_tokens_details': None, 'prompt_time': 0.002593641, 'prompt_tokens_details': None, 'queue_time': 0.046387884, 'total_time': 0.020007729}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4f2a-69cf-7e01-8952-ec3963de4442-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 6, 'total_tokens': 60})

In [21]:
model.invoke([HumanMessage("Hi, my name is kajal"),
  ])

model.invoke([HumanMessage("Hi, what was my name"),
  ])

AIMessage(content="I'm a large language model, I don't have the ability to retain information about individual users or their previous conversations. Each time you interact with me, it's a new conversation and I don't have any prior knowledge or context about you.\n\nIf you'd like to share your name with me, I'd be happy to chat with you and get to know you better!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 41, 'total_tokens': 117, 'completion_time': 0.165015639, 'completion_tokens_details': None, 'prompt_time': 0.001969282, 'prompt_tokens_details': None, 'queue_time': 0.047163508, 'total_time': 0.166984921}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4f3e-4dd7-7c22-9fbc-70f0a373e342-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 76, '

Message History

We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [7]:
##message history
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [8]:
config={"configurable": {"session_id": "user1"}}
with_message_history.invoke([HumanMessage(content="hi my name is kajal")],config=config)

AIMessage(content="Hello Kajal. It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 42, 'total_tokens': 70, 'completion_time': 0.02522083, 'completion_tokens_details': None, 'prompt_time': 0.001938645, 'prompt_tokens_details': None, 'queue_time': 0.04682697, 'total_time': 0.027159475}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4f2a-8b8d-7d61-a861-a0ef6bad9451-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 28, 'total_tokens': 70})

In [9]:
with_message_history.invoke([HumanMessage(content="tell me about myself?")],config=config)

AIMessage(content='I\'ll do my best to provide some general information about yourself based on your name. However, please note that this is just a guess, and the actual information may vary.\n\nYour name, Kajal, is of Indian origin. In Hindi, "Kajal" means "kohl" or "eye makeup." It\'s a popular name in many parts of the world, especially in India and other South Asian countries.\n\nSome general traits associated with people named Kajal include:\n\n1. Creativity: People with this name are often known for their artistic side, creative expression, and imaginative thinking.\n2. Confidence: Kajal is a name that exudes confidence and self-assurance, which can help individuals with this name to stand out in their personal and professional lives.\n3. Emotional intelligence: Those with this name are often empathetic and understanding, with a strong connection to their emotions and the emotions of those around them.\n4. Determination: Kajal is a name that suggests determination and perseveranc

Prompt templates

Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant answers all the questions to the best of your ability."),
        MessagesPlaceholder(variable_name="input")
    ]
)
chain=prompt | model

with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input"
)

In [11]:
with_message_history.invoke(
    {"input": [HumanMessage(content="hi my name is abhi")]},
    config=config
)

AIMessage(content='Hello Abhi. It\'s nice to meet you. As we did earlier, I\'ll share some general information about the name Abhi.\n\nThe name Abhi is of Indian origin, commonly used in many parts of the world, especially in India, Nepal, and other South Asian countries. It\'s derived from the Sanskrit word "Abhi," which means "now" or "at this time." It\'s also a shortened form of the name Abhinav, which means "new" or "novel."\n\nSome general traits associated with people named Abhi include:\n\n1. Energetic: Abhi is a name that suggests energy, enthusiasm, and a lively personality.\n2. Adventurous: People with this name are often known for their love of adventure, trying new things, and exploring new experiences.\n3. Confident: Abhi is a name that exudes self-assurance and confidence, which can help individuals with this name to take on new challenges and achieve their goals.\n4. Friendly: Those with this name are often known for their friendly and outgoing nature, making them popul

Managing the Conversation History

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

In [12]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

In [17]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage


chain=(
    RunnablePassthrough.assign(input=itemgetter("input")|trimmer)| prompt | model
)

messages = [
    SystemMessage(content="You are helpful"),
    HumanMessage(content="Hi"),
    AIMessage(content="Hello!"),
    HumanMessage(content="Tell me a joke")
]

response = chain.invoke({"input": messages})
print(response)

content='What do you call a fake noodle?\n\nAn impasta.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 71, 'total_tokens': 85, 'completion_time': 0.024269044, 'completion_tokens_details': None, 'prompt_time': 0.004315804, 'prompt_tokens_details': None, 'queue_time': 0.047619226, 'total_time': 0.028584848}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d4f2e-424a-7801-aaf4-6916754a23f2-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 71, 'output_tokens': 14, 'total_tokens': 85}


In [19]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input")

config={"configurable": {"session_id": "user1"}}
with_message_history.invoke({"input": messages}, config=config)
with_message_history.invoke({"input": "what was the last joke?"}, config=config)

AIMessage(content='The last joke I told you was about a man asking a librarian for books on Pavlov\'s dogs and Schrödinger\'s cat, and the librarian responding that it "rings a bell, but I\'m not sure if it\'s here or not."', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 124, 'total_tokens': 178, 'completion_time': 0.064356575, 'completion_tokens_details': None, 'prompt_time': 0.007179623, 'prompt_tokens_details': None, 'queue_time': 0.046218076, 'total_time': 0.071536198}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4f3c-4853-7d81-a73d-26da759763a7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 124, 'output_tokens': 54, 'total_tokens': 178})